In [52]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.ensemble import GradientBoostingClassifier 
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score, roc_auc_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

In [53]:
df = pd.read_csv('f1_results_2025.csv')

In [54]:
df.head()

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished


In [55]:
df['final_position'] = pd.to_numeric(df['final_position'], errors='coerce')

In [56]:
df

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
274,2025,14,Hungarian Grand Prix,2025-08-03,Hungaroring,Esteban Ocon,French,Haas F1 Team,17,16,0.0,69,+45.258,1:21.916,Lapped
275,2025,14,Hungarian Grand Prix,2025-08-03,Hungaroring,Yuki Tsunoda,Japanese,Red Bull,20,17,0.0,69,+46.943,1:21.180,Lapped
276,2025,14,Hungarian Grand Prix,2025-08-03,Hungaroring,Franco Colapinto,Argentine,Alpine F1 Team,14,18,0.0,69,+47.370,1:20.827,Lapped
277,2025,14,Hungarian Grand Prix,2025-08-03,Hungaroring,Pierre Gasly,French,Alpine F1 Team,16,19,0.0,69,+56.344,1:21.433,Lapped


In [57]:
preferred = [
    'fastest_lap'
]
auto = [c for c in df.columns if 'lap' in c.lower() and ('fast' in c.lower() or 'fastest' in c.lower())]
for p in preferred:
    if p in df.columns and p not in auto:
        auto.append(p)
col = auto[0] if auto else None


s = (df[col]
        .astype('string')
        .fillna('')
        .str.strip()
        .str.lower()
        .str.replace(' ', '', regex=False))

s = s.str.replace(r'(?<=\d)m(?=\d)', ':', regex=True)
s = s.str.replace(r's$', '', regex=True)

colon_count = s.str.count(':')
with_colon = s.where(colon_count.gt(0))
base = with_colon.fillna('')

# Convert timedeltas to seconds without using .dt (avoid TimedeltaIndex .dt error)
td = pd.to_timedelta(
    np.where(colon_count.eq(1), '00:' + base, base),
    errors='coerce'
)
secs_colon = (td / pd.Timedelta(seconds=1)).astype(float)

secs_no_colon = pd.to_numeric(s.where(colon_count.eq(0)), errors='coerce')

mask = colon_count.gt(0).to_numpy()
a = np.asarray(secs_colon, dtype=float)
b = np.asarray(secs_no_colon, dtype=float)

df['fastest_lap_seconds'] = np.where(mask, a, b).astype('float64')
parsed = int(np.isfinite(df['fastest_lap_seconds']).sum())
total = len(df)
print(f"Created df['fastest_lap_seconds'] from column '{col}' ({parsed}/{total} parsed).")

Created df['fastest_lap_seconds'] from column 'fastest_lap' (264/279 parsed).


In [58]:
df.head()

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished,82.167
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished,83.081
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished,85.065
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished,84.901
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished,84.597


In [59]:
df.head()

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
0,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Lando Norris,British,McLaren,1,1,25.0,57,1:42:06.304,1:22.167,Finished,82.167
1,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Max Verstappen,Dutch,Red Bull,3,2,18.0,57,+0.895,1:23.081,Finished,83.081
2,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,George Russell,British,Mercedes,4,3,15.0,57,+8.481,1:25.065,Finished,85.065
3,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Andrea Kimi Antonelli,Italian,Mercedes,16,4,12.0,57,+10.135,1:24.901,Finished,84.901
4,2025,1,Australian Grand Prix,2025-03-16,Albert Park Grand Prix Circuit,Alexander Albon,Thai,Williams,6,5,10.0,57,+12.773,1:24.597,Finished,84.597


In [60]:
df.tail(60)

,season_year,race_round_number,race_name,race_date,circuit_name,driver_name,driver_nationality,constructor_name,grid_position,final_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
219,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Lando Norris,British,McLaren,3,1,25.0,52,1:37:15.735,1:29.734,Finished,89.734
220,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Oscar Piastri,Australian,McLaren,2,2,18.0,52,+6.812,1:29.337,Finished,89.337
221,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Nico Hülkenberg,German,Sauber,19,3,15.0,52,+34.742,1:30.933,Finished,90.933
222,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Lewis Hamilton,British,Ferrari,5,4,12.0,52,+39.812,1:30.016,Finished,90.016
223,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Max Verstappen,Dutch,Red Bull,1,5,10.0,52,+56.781,1:30.179,Finished,90.179
224,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Pierre Gasly,French,Alpine F1 Team,8,6,8.0,52,+59.857,1:30.751,Finished,90.751
225,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Lance Stroll,Canadian,Aston Martin,17,7,6.0,52,+1:00.603,1:32.088,Finished,92.088
226,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Alexander Albon,Thai,Williams,13,8,4.0,52,+1:04.135,1:30.047,Finished,90.047
227,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,Fernando Alonso,Spanish,Aston Martin,7,9,2.0,52,+1:05.858,1:30.353,Finished,90.353
228,2025,12,British Grand Prix,2025-07-06,Silverstone Circuit,George Russell,British,Mercedes,4,10,1.0,52,+1:10.674,1:30.869,Finished,90.869


In [64]:

cand = lambda *xs: next((c for c in xs if c in df.columns), None)
col_driver = cand('driver_name', 'driver', 'Driver', 'driverId', 'driver_id')
col_final  = cand('final_position', 'position', 'Result')
col_points = cand('points', 'Points')
col_date   = cand('race_date', 'date', 'Date')
col_season = cand('season', 'Year')
col_round  = cand('round', 'Round')
col_race   = cand('race_name', 'grand_prix', 'Grand Prix')

if col_driver is None or col_final is None:
    raise KeyError('Required columns not found: driver and final_position')

work = df.copy()
work[col_final] = pd.to_numeric(work[col_final], errors='coerce')
if col_points:
    work[col_points] = pd.to_numeric(work[col_points], errors='coerce')

# Ensure season is available (derive from date if needed)
if col_season is None and col_date:
    work[col_date] = pd.to_datetime(work[col_date], errors='coerce', dayfirst=True)
    work['__season_tmp__'] = work[col_date].dt.year
    col_season = '__season_tmp__'

# Identify 2025 grid drivers
if col_season is None:
    raise KeyError('No season/year information available to identify 2025 grid drivers.')

grid_mask = work[col_season] == 2025
grid_drivers = (
    work.loc[grid_mask & work[col_driver].notna(), col_driver]
        .dropna()
        .unique()
        .tolist()
)

# Nothing to compute if none found
if not grid_drivers:
    raise ValueError('No drivers found for season 2025 in the dataset.')

# Sort chronologically
if col_date:
    # If date col exists but not yet parsed
    if not np.issubdtype(work[col_date].dtype, np.datetime64):
        work[col_date] = pd.to_datetime(work[col_date], errors='coerce', dayfirst=True)
    sort_cols = [col_date]
    if col_race:
        sort_cols.append(col_race)
elif col_season and col_round:
    sort_cols = [col_season, col_round]
else:
    sort_cols = [work.index]

work = work.sort_values(sort_cols).reset_index(drop=True)

# Keep only 2025 grid drivers
work = work[work[col_driver].isin(grid_drivers)].copy()

# Take last 3 races per driver
by = work[col_driver]
last3 = work.groupby(by, group_keys=False).tail(5)

# Aggregate form metrics
form_2025 = last3.groupby(col_driver).agg(
    races=(col_final, 'count'),
    avg_finish=(col_final, 'mean'),
    median_finish=(col_final, 'median'),
    best_finish=(col_final, 'min'),
    top10_rate=(col_final, lambda s: np.mean((s <= 10).astype(float))),
    podium_rate=(col_final, lambda s: np.mean((s <= 3).astype(float))),
    win_rate=(col_final, lambda s: np.mean((s == 1).astype(float))),
)
if col_points:
    pts = last3.groupby(col_driver)[col_points].agg(avg_points='mean', sum_points='sum')
    form_2025 = form_2025.join(pts)

# Order: best average finish, then best finish
form_2025 = form_2025.sort_values(['avg_finish', 'avg_points'])

print(f"Current form for 2025 grid (last 3 GPs): Drivers = {len(form_2025)}")
form_2025

Current form for 2025 grid (last 3 GPs): Drivers = 21


C:\Users\Divyansh\AppData\Local\Temp\ipykernel_26484\1966665082.py:20: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  work[col_date] = pd.to_datetime(work[col_date], errors='coerce', dayfirst=True)


,races,avg_finish,median_finish,best_finish,top10_rate,podium_rate,win_rate,avg_points,sum_points
driver_name,,,,,,,,,
Oscar Piastri,5,2.2,2.0,1,1.0,0.8,0.2,18.2,91.0
Lando Norris,5,4.6,1.0,1,0.8,0.8,0.6,18.6,93.0
George Russell,5,4.8,5.0,1,1.0,0.4,0.2,12.2,61.0
Charles Leclerc,5,5.8,4.0,3,0.8,0.4,0.0,10.4,52.0
Lewis Hamilton,5,6.6,6.0,4,0.8,0.0,0.0,7.6,38.0
Max Verstappen,5,7.6,5.0,2,0.8,0.2,0.0,8.4,42.0
Nico Hülkenberg,5,9.0,9.0,3,0.6,0.2,0.0,4.2,21.0
Fernando Alonso,5,9.0,7.0,5,0.8,0.0,0.0,4.8,24.0
Gabriel Bortoleto,5,11.0,9.0,6,0.6,0.0,0.0,2.8,14.0


In [75]:

features = df.copy()

driver_form = form_2025.reset_index()


In [76]:
# Split data for training/testing
from sklearn.model_selection import train_test_split

# Define X (features) and y (target - race winner or position)
X = features.drop(['final_position', 'driver_name', 'race_name', 'race_date'], axis=1)
y = features[['final_position','grid_position']]  # or binary target (1 for winner, 0 for non-winner)

# Create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

In [77]:
X.head()


,season_year,race_round_number,circuit_name,driver_nationality,constructor_name,grid_position,points,laps_completed,race_time,fastest_lap,status,fastest_lap_seconds
0,2025,1,Albert Park Grand Prix Circuit,British,McLaren,1,25.0,57,1:42:06.304,1:22.167,Finished,82.167
1,2025,1,Albert Park Grand Prix Circuit,Dutch,Red Bull,3,18.0,57,+0.895,1:23.081,Finished,83.081
2,2025,1,Albert Park Grand Prix Circuit,British,Mercedes,4,15.0,57,+8.481,1:25.065,Finished,85.065
3,2025,1,Albert Park Grand Prix Circuit,Italian,Mercedes,16,12.0,57,+10.135,1:24.901,Finished,84.901
4,2025,1,Albert Park Grand Prix Circuit,Thai,Williams,6,10.0,57,+12.773,1:24.597,Finished,84.597


NameError: name 'Y' is not defined